# 1. My lane as an ML task

**Lane:** CTR / Engagement Opportunity Scoring

**ML Task Type:** Ranking (with an opportunity score)

This project is framed as a **ranking problem** rather than a binary classification task. Instead of predicting whether a page belongs to a fixed class, the goal is to assign each page an Opportunity Score and rank pages according to their potential for review. The highest-ranked pages become candidates for investigation by the SEO team.

# 2. Target or Proxy

### Target / Proxy

The final target will likely be a proxy for "review opportunity" because no direct label exists that says whether a page deserved review.

An initial proxy for review opportunity combines:
* high search visibility (impressions_90d)
* relatively low CTR (ctr)
* relatively low engagement (engagement_rate)
* declining performance (trend_direction, trend_pct)
* content freshness (days_since_last_update)

Pages exhibiting this combination may represent opportunities where improving metadata or content could increase performance.

The exact proxy may evolve after exploratory data analysis in later weeks.

# 3. Success Metric

The success of the ranking system will be evaluated using ranking-oriented metrics such as **Precision@K** or similar top-K evaluation, since the objective is to recommend the most valuable pages for review rather than classify every page correctly.

Secondary evaluation may include comparing the proposed ranking against a simple rule-based baseline.

# 4. The unit of analysis, as a real dataframe

Each row represents one webpage together with aggregated search performance and engagement measurements.

The model will assign an Opportunity Score to each row independently before ranking all pages.

In [1]:
import pandas as pd
from pathlib import Path

csv_path = Path("../../data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(csv_path)

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier,age_tier_order,days_since_last_update,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,NaN,gemini-2.5-flash,3803,29,22,17,16,1,0,1,88,13,578,2,2,987,13,9,187,181-365,5,20,0-30,2000-3500,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,NaN,gemini-3-flash-preview,15320,7,10,9,9,0,0,1,88,9,2501,2,3,5915,1,2,445,365+,6,25,0-30,2000-3500,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,NaN,gemini-2.5-flash,12581,11,14,11,11,0,0,4,88,11,2382,1,1,6089,3,3,141,91-180,4,20,0-30,3500+,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,NaN,NaN,11751,58,87,78,75,1,0,3,88,51,3626,22,35,4206,17,26,463,365+,6,22,0-30,NaN,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,NaN,gemini-3-flash-preview,19140,24,177,145,144,0,0,43,88,33,4211,10,14,6452,2,9,263,181-365,5,14,0-30,2000-3500,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [2]:
df[
    [
        "impressions_90d",
        "ctr",
        "engagement_rate"
    ]
].head()

,impressions_90d,ctr,engagement_rate
0,3803,0.76,5.88
1,15320,0.05,0.00
2,12581,0.09,0.00
3,11751,0.49,1.28
4,19140,0.13,0.00


These columns illustrate the signals that may be combined into an Opportunity Score. The final proxy target will be refined after further exploratory analysis.

In [3]:
df["review_opportunity_proxy"] = (
    (df["impressions_90d"] > 1000)
    & (df["ctr"] < 1.0)
    & (df["engagement_rate"] < 5)
)

df[
    [
        "impressions_90d",
        "ctr",
        "engagement_rate",
        "review_opportunity_proxy"
    ]
].head()

,impressions_90d,ctr,engagement_rate,review_opportunity_proxy
0,3803,0.76,5.88,False
1,15320,0.05,0.00,True
2,12581,0.09,0.00,True
3,11751,0.49,1.28,True
4,19140,0.13,0.00,True


This is not the final target used for modeling. It is an initial proxy that demonstrates how a review opportunity label could be defined using observable signals. The definition will likely evolve after exploratory data analysis and leakage checks.

# 5. Why ML beats a fixed rule here

A simple rule such as
```
CTR < 1%
```
would ignore many important factors.

For example:
* A page with 50 impressions and 0.5% CTR is very different from a page with 50,000 impressions and 0.5% CTR.
* CTR also depends on average search position.
* Engagement metrics provide additional evidence about user experience.

Machine learning can combine these interacting signals to produce a more useful ranking than a single manually chosen threshold.

**Baseline:** Rank pages with CTR below 1% by descending impressions.

The proposed ML ranking should outperform this heuristic by incorporating additional signals such as position, engagement, freshness, and performance trends.

# 6. Self Check

- [x] Names the ML task type
- [x] Names target or proxy
- [x] Names success metric
- [x] Shows unit of analysis as a real dataframe
- [x] Explains why this is an ML/analysis problem and not just a rule
- [x] Ties output to a real content action